In [ ]:
# %% Deep learning - Section 19.178
#    Code challenge 27: softcode internal parameters
#
#    1) Start from code from video 19.177 (gaussian synthetic dataset)
#    2) Softcode the number of channels in the convolutional layer 3 (first) and
#       6 (second)
#    3) Softcode (anisotropic) kernel size to (3,2)
#    4) Softcode (anisotropic) kernel size to (2,3)
#    5) Softcode the number of inputs into fc1
#    6) Change the feature map plotting code to softcode the number of rows
#       according to the feature map size

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [ ]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [84]:
# %% Generate data

# 2D Gaussian params
n_per_class = 1000
n_classes   = 2
img_size    = 151
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

widths      = [1.8,2.4]

# Preallocate tensors for images (N,channels,size,size) and labels (N)
images  = torch.zeros( n_classes*n_per_class,1,img_size,img_size )
labels  = torch.zeros( n_classes*n_per_class )

# Generate images
for i in range(n_classes*n_per_class):

    # Gaussian with random center offset (remainder trick for width, all even
    # images go into category 0, all odd images go into category 1)
    c = 2*np.random.randn(2)
    G = np.exp( -((X-c[0])**2 + (Y-c[1])**2 ) / (2*widths[i%2]**2) )

    # Layer some noise
    G = G + np.random.randn(img_size,img_size)/5

    # Add to tensor
    images[i,:,:,:] = torch.tensor(G).view(1,img_size,img_size)
    labels[i]       = i%2

labels = labels[:,None]


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(3,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):

    pic = np.random.randint(2*n_per_class)
    G   = np.squeeze( images[pic,:,:] )

    ax.imshow(G,vmin=-1,vmax=1,cmap='jet')
    ax.set_title('Class %s'%int(labels[pic].item()))
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig('figure32_code_challenge_27.png')
plt.show()
files.download('figure32_code_challenge_27.png')


In [85]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [86]:
# %% Function to generate the model

def gen_model(img_size,conv_chan1=3,conv_chan2=6,k=(3,2),s=(2,3)):

    class mnist_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Convolution layer 1
            # output size: s = ((img+2*p-k)/s) + 1
            # post-pooling: s/2
            self.conv1 = nn.Conv2d(1,conv_chan1,kernel_size=k,stride=s,padding=1)

            height = np.floor( (img_size+2*self.conv1.padding[0]-k[0])/s[0] ) + 1
            height = np.floor( height/2 )
            width  = np.floor( (img_size+2*self.conv1.padding[0]-k[1])/s[1] ) + 1
            width  = np.floor( width/2 )

            # Convolution layer 2
            self.conv2 = nn.Conv2d(conv_chan1,conv_chan2,kernel_size=k,stride=s,padding=1)

            height = np.floor( (height+2*self.conv2.padding[0]-k[0])/s[0] ) + 1
            height = np.floor( height/2 )
            width  = np.floor( (width+2*self.conv2.padding[0]-k[1])/s[1] ) + 1
            width  = np.floor( width/2 )

            # Fully connected layer
            height = int(height)
            width  = int(width)
            chans  = int(conv_chan2)
            size   = height * width * chans

            self.fc1 = nn.Linear(size,50)

            # Output layer
            self.output = nn.Linear(50,1)

        def forward(self,x):

            # Adapt to export
            # MaxPool and ReLu on convolution layer 1
            conv1_act = F.relu(self.conv1(x))
            x         = F.avg_pool2d(conv1_act,(2,2))

            # MaxPool and ReLu on convolution layer 2
            conv2_act = F.relu(self.conv2(x))
            x         = F.avg_pool2d(conv2_act,(2,2))

            # Vectorise for linear layer
            x = torch.flatten(x,start_dim=1)

            # Linear and output layers
            x = F.relu(self.fc1(x))
            x = self.output(x)

            return x,conv1_act,conv2_act

    # Create model instance
    CNN = mnist_CNN()

    # Loss function
    loss_fun = nn.BCEWithLogitsLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

# test the model with one batch
CNN,loss_fun,optimizer = gen_model( img_size,
                                    conv_chan1=3,
                                    conv_chan2=6,
                                    k=(3,2),
                                    s=(2,3) )

# test that the model runs and can compute a loss
X,y = next(iter(train_loader))
yHat,feat_map1,feat_map2 = CNN(X)
loss = loss_fun(yHat,y)

# check sizes of outputs
print('Predicted category:')
print(yHat.shape)
print('\nFeature map after conv1')
print(feat_map1.shape)
print('\nFeature map after conv2')
print(feat_map2.shape)


In [ ]:
# %% Check all the parameters in the model

summary(CNN,(1,img_size,img_size))


In [89]:
# %% Function to train the model

def train_model(img_size,conv_chan1,conv_chan2,k,s):

    # Parameters, model instance, inizialise vars
    num_epochs = 10
    CNN,loss_fun,optimizer = gen_model( img_size,
                                        conv_chan1=conv_chan1,
                                        conv_chan2=conv_chan2,
                                        k=k,
                                        s=s )

    train_losses = []
    test_losses  = []
    train_acc    = []
    test_acc     = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Forward propagation and loss (only select 1st output)
            yHat = CNN(X)[0]
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            batch_loss.append(loss.item())
            batch_acc.append( torch.mean( ((yHat>0)==y).float() ).item() )

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Test accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))
            yHat = CNN(X)[0]
            loss = loss_fun(yHat,y)

            test_acc.append( 100*torch.mean( ((yHat>0)==y).float() ).item() )
            test_losses.append(loss.item())

        CNN.train()

    return train_acc,test_acc,train_losses,test_losses,CNN


In [90]:
# %% Run the model

# Takes ~30 secs
conv_chan1=3
conv_chan2=6
k=(3,2)
s=(2,3)

train_acc,test_acc,train_losses,test_losses,CNN = train_model(img_size,conv_chan1,conv_chan2,k,s)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_losses,'s-',label='Train')
ax[0].plot(test_losses,'o-',label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')

ax[1].plot(train_acc,'s-',label='Train')
ax[1].plot(test_acc,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model test accuracy: {test_acc[-1]:.2f}%')
ax[1].legend()

plt.savefig('figure33_code_challenge_27.png')
plt.show()
files.download('figure33_code_challenge_27.png')


In [ ]:
# %% Plotting

# Pass test data through model
X,y = next(iter(test_loader))
yHat,feat_map1,feat_map2 = CNN(X)

# Plotting
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(conv_chan1+1,10,figsize=(1.5*phi*6,6))

for pic_i in range(10):

    # Show the original picture
    img = X[pic_i,0,:,:].detach()
    axs[0,pic_i].imshow(img,cmap='jet',vmin=0,vmax=1)
    axs[0,pic_i].axis('off')
    axs[0,pic_i].text(2,2,'T=%s'%int(y[pic_i].item()),ha='left',va='top',color='w',fontweight='bold')

    for feat_i in range(conv_chan1):

        # Extract the feature map from this image
        img = feat_map1[pic_i,feat_i,:,:].detach()
        axs[feat_i+1,pic_i].imshow(img,cmap='plasma',vmin=0,vmax=torch.max(img)*.9)
        axs[feat_i+1,pic_i].axis('off')
        axs[feat_i+1,pic_i].text(-5,45,feat_i,ha='right') if pic_i==0 else None

# plt.tight_layout()
plt.suptitle('First set of feature map activations for 10 test images',x=.5,y=1.01)

plt.savefig('figure34_code_challenge_27.png')
plt.show()
files.download('figure34_code_challenge_27.png')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(conv_chan2+1,10,figsize=(1.5*phi*6,6))

for pic_i in range(10):

    # Show the original picture
    img = X[pic_i,0,:,:].detach()
    axs[0,pic_i].imshow(img,cmap='jet',vmin=0,vmax=1)
    axs[0,pic_i].axis('off')
    axs[0,pic_i].text(2,2,'T=%s'%int(y[pic_i].item()),ha='left',va='top',color='w',fontweight='bold')

    for feat_i in range(conv_chan2):

        # Extract the feature map from this image
        img = feat_map2[pic_i,feat_i,:,:].detach()
        axs[feat_i+1,pic_i].imshow(img,cmap='plasma',vmin=0,vmax=torch.max(img)*.9)
        axs[feat_i+1,pic_i].axis('off')
        axs[feat_i+1,pic_i].text(-5,5,feat_i,ha='right') if pic_i==0 else None

# plt.tight_layout()
plt.suptitle('First set of feature map activations for 10 test images',x=.5,y=1.01)

plt.savefig('figure35_code_challenge_27.png')
plt.show()
files.download('figure35_code_challenge_27.png')


In [36]:
# %% Exercise 1
#    Notice that the model outputs the feature map activation *before* pooling. Do you agree with that decision? Modify
#    the code to export the feature activation map *after* pooling is applied. Does that qualitatively change the results?

# Different data because it's a different run, but in theory the only thing that
# changes is the resolution of the feature maps, which is lower after pooling,
# in this case we even use mean pooling, which preserve some more smoothness

# Function to generate the model
def gen_model(conv_chan1=3,conv_chan2=6,k=(3,2),s=(2,3)):

    class mnist_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Convolution layer 1
            # output size: s = ((img+2*p-k)/s) + 1
            # post-pooling: s/2
            self.conv1 = nn.Conv2d(1,conv_chan1,kernel_size=k,stride=s,padding=1)

            height = np.floor( (91+2*self.conv1.padding[0]-k[0])/s[0] ) + 1
            height = np.floor( height/2 )
            width  = np.floor( (91+2*self.conv1.padding[0]-k[1])/s[1] ) + 1
            width  = np.floor( width/2 )

            # Convolution layer 2
            self.conv2 = nn.Conv2d(conv_chan1,conv_chan2,kernel_size=k,stride=s,padding=1)

            height = np.floor( (height+2*self.conv2.padding[0]-k[0])/s[0] ) + 1
            height = np.floor( height/2 )
            width  = np.floor( (width+2*self.conv2.padding[0]-k[1])/s[1] ) + 1
            width  = np.floor( width/2 )

            # Fully connected layer
            height = int(height)
            width  = int(width)
            chans  = int(conv_chan2)
            size   = height * width * chans

            self.fc1 = nn.Linear(size,50)

            # Output layer
            self.output = nn.Linear(50,1)

        def forward(self,x):

            # Adapt to export
            # MaxPool and ReLu on convolution layer 1
            conv1_act_pre  = self.conv1(x)
            conv1_act      = F.relu(conv1_act_pre)
            conv1_act_post = F.avg_pool2d(conv1_act,(2,2))

            # MaxPool and ReLu on convolution layer 2
            conv2_act_pre  = self.conv2(conv1_act_post)
            conv2_act      = F.relu(conv2_act_pre)
            conv2_act_post = F.avg_pool2d(conv2_act,(2,2))

            # Vectorise for linear layer
            x = torch.flatten(conv2_act_post,start_dim=1)

            # Linear and output layers
            x = F.relu(self.fc1(x))
            x = self.output(x)

            return x,conv1_act_post,conv2_act_post

    # Create model instance
    CNN = mnist_CNN()

    # Loss function
    loss_fun = nn.BCEWithLogitsLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [57]:
# %% Exercise 2
#    I mentioned in the lecture that there is a trade-off between soft- and hard-coding. One variable that is hardcoded
#    in the model is the image size (91x91). Does that matter? Change the size of the images (variable imgSize at the top
#    of the notebook) to 89x89. Does that break the model? Try again, changing imgSize=81. Does that break the model?
#    Why do get these results? Finally, modify the model code so that the image size is soft-coded.

# Changing the size make the code crash because the matrix sizes don't add up;
# however that can be easily fixed by passing the image size parameter to the
# model class

# See code above
